In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_42_try_0.pickle

In [ ]:
%%cudf.pandas.profile
### cell 43 ###

### cell 43 (optimized for cudf)

def combine_subset_of_data_from_multiple_years_55(
    question_of_interest, x_axis_title, include_2017=False
):
    # Map each year to its GPU-backed responses dataframe
    mapping = {
        '2017': responses_df_2017,
        '2018': responses_df_2018_cell10,
        '2019': responses_df_2019_cell10,
        '2020': responses_df_2020,
        '2021': responses_df_2021,
        '2022': responses_df_2022_cell10
    }
    # Pick the years in order
    years = ['2017','2018','2019','2020','2021','2022'] if include_2017 else ['2018','2019','2020','2021','2022']

    # Concatenate with a 'year' column
    df_all = pd.concat(
        [mapping[yr][[question_of_interest]].assign(year=yr) for yr in years],
        ignore_index=True
    )

    # Fill nulls with a placeholder and cast to categorical for GPU-friendly groupby
    df_all[question_of_interest] = (
        df_all[question_of_interest]
        .fillna('<Missing>')
        .astype('category')
    )
    df_all['year'] = df_all['year'].astype('category')

    # GPU-accelerated groupby and count
    grouped = (
        df_all
        .groupby(['year', question_of_interest])
        .size()
        .reset_index(name='count')
    )

    # Compute percentage within each year (all GPU)
    grouped['percentage'] = (
        grouped['count'] * 100
        / grouped.groupby('year')['count'].transform('sum')
    ).round(1)

    # Rename and select columns
    result = grouped.rename(
        columns={question_of_interest: x_axis_title}
    )[['percentage', 'year', x_axis_title]]

    return result

# Re-run aggregation
question_of_interest_cell55 = (
    'Select the title most similar to your current role (or most recent title if retired): - Selected Choice'
)
title_for_x_axis = title_for_x_axis_cell43

job_df_combined = combine_subset_of_data_from_multiple_years_55(
    question_of_interest_cell55,
    title_for_x_axis,
    include_2017=False
)

# Standardize and filter (all GPU)
job_df_combined[title_for_x_axis] = (
    job_df_combined[title_for_x_axis]
    .replace(
        {'Data Analyst (Business, Marketing, Financial, Quantitative, etc)': 'Data Analyst'}
    )
)

job_df_combined = job_df_combined[
    job_df_combined[title_for_x_axis].isin([
        'Data Analyst', 'Data Engineer', 'Data Scientist',
        'Research Scientist', 'Software Engineer'
    ])
]

job_df_combined.info()
